# Hyperparameter Tuning: Grid, Random & Optuna
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/algorithms/hyperparameter_tuning.ipynb)

Hyperparameters are settings chosen before training. This notebook compares exhaustive GridSearchCV, smarter RandomizedSearchCV, and Bayesian optimization with Optuna.

**Covered:** pipelines + grids, randomized search, Optuna study, nested-CV warning.

## 1. Baseline + GridSearchCV

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, random_state=42)

pipe = Pipeline([("scaler", StandardScaler()), ("svm", SVC())])
grid = {"svm__C": [0.1, 1, 10, 100], "svm__gamma": ["scale", 0.001, 0.01, 0.1]}

gs = GridSearchCV(pipe, grid, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                  scoring="roc_auc", n_jobs=-1)
gs.fit(Xtr, ytr)
print("best params:", gs.best_params_)
print(f"best CV AUC: {gs.best_score_:.4f}   test AUC: {gs.score(Xte, yte):.4f}")

Grid size = len(C) x len(gamma) = 16 fits x 5 folds. Doubles with every new value added.

## 2. RandomizedSearchCV - better budget use

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.ensemble import RandomForestClassifier

rpipe = Pipeline([("rf", RandomForestClassifier(random_state=42))])
dist = {"rf__n_estimators": randint(50, 500),
        "rf__max_depth": randint(2, 20),
        "rf__min_samples_split": randint(2, 10),
        "rf__max_features": ["sqrt", "log2", None]}

rs = RandomizedSearchCV(rpipe, dist, n_iter=40, cv=5,
                        scoring="roc_auc", random_state=42, n_jobs=-1)
rs.fit(Xtr, ytr)
print("best params:", rs.best_params_)
print(f"best CV AUC: {rs.best_score_:.4f}   test AUC: {rs.score(Xte, yte):.4f}")

## 3. Optuna - Bayesian optimization

In [ ]:
!pip install -q optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "C": trial.suggest_float("C", 1e-2, 1e2, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 1e-1, log=True),
    }
    pipe_o = Pipeline([("scaler", StandardScaler()), ("svm", SVC(**params))])
    return cross_val_score(pipe_o, Xtr, ytr, cv=5, scoring="roc_auc").mean()

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40)
print("best value :", round(study.best_value, 4))
print("best params:", study.best_params_)

In [ ]:
from sklearn.model_selection import cross_val_score
best_pipe = Pipeline([("scaler", StandardScaler()),
                      ("svm", SVC(**study.best_params))])
print(f"Optuna-tuned test AUC: {cross_val_score(best_pipe, Xte, yte, cv=5, scoring='roc_auc').mean():.4f}")

## Practical guidance
| Method | When |
|---|---|
| GridSearchCV | <= 3 hyperparams, small discrete grids |
| RandomizedSearchCV | wider spaces, fixed compute budget |
| Optuna / Bayesian | expensive models, continuous spaces |

**Never tune on the test set** - wrap the whole search in an outer CV loop (nested CV) for unbiased estimates.